In [0]:
df = spark.sql("""
                 select distinct identifier,NameShop,GrundWebsperre,Marke,url_prefix,Keywords,seo_slug,item_id,category_caption_level_2,family,Inhaltsstoffe_new,Warnungen_new,category,Langbeschreibung_new,descriptions,Name_AX_embedding,Inhaltsstoffe_new_embedding,Warnungen_new_embedding,category_embedding,descriptions_embedding  
                 from datascience.at_product_details_individual_embeddings
                 """)

In [0]:
display(df)

In [0]:
import numpy as np
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, FloatType

# Define your custom weights for each embedding column
# Adjust these weights according to your preference
w_name = 0.4
w_inhaltsstoffe = 0.2
w_warnungen = 0.05
w_category = 0.2
w_descriptions = 0.15

# Define a UDF that combines the embeddings
def combine_embeddings(name_emb, inhaltsstoffe_emb, warnungen_emb, category_emb, descriptions_emb):
    # Check for None values; you might want to handle missing embeddings appropriately
    if None in (name_emb, inhaltsstoffe_emb, warnungen_emb, category_emb, descriptions_emb):
        return None
    # Convert lists to numpy arrays
    arr_name = np.array(name_emb, dtype=float)
    arr_inhaltsstoffe = np.array(inhaltsstoffe_emb, dtype=float)
    arr_warnungen = np.array(warnungen_emb, dtype=float)
    arr_category = np.array(category_emb, dtype=float)
    arr_descriptions = np.array(descriptions_emb, dtype=float)
    
    # Perform element-wise weighted sum
    combined = (w_name * arr_name + 
                w_inhaltsstoffe * arr_inhaltsstoffe + 
                w_warnungen * arr_warnungen + 
                w_category * arr_category + 
                w_descriptions * arr_descriptions)
    
    return combined.tolist()

# Register the UDF
combine_udf = udf(combine_embeddings, ArrayType(FloatType()))

# Assuming your dataframe is called df, create a new column for the combined embedding
df = df.withColumn(
    "combined_embedding",
    combine_udf(
        col("Name_AX_embedding"),
        col("Inhaltsstoffe_new_embedding"),
        col("Warnungen_new_embedding"),
        col("category_embedding"),
        col("descriptions_embedding")
    )
)

# Now df has a new column "combined_embedding" that you can use for similarity computations


In [0]:
display(df)

In [0]:
pip install faiss-cpu umap-learn

In [0]:
import numpy as np
import time
import faiss
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [0]:
df_pandas = df.select("identifier","item_id", "NameShop", "combined_embedding").toPandas()

In [0]:
# Convert the "combined_embedding" column (list of floats) into a NumPy array.
embeddings_list = df_pandas['combined_embedding'].tolist()
embeddings = np.array(embeddings_list, dtype='float32')
num_vectors, d = embeddings.shape  # d should be 512

In [0]:
# Create lists for item_ids and name_ax values (used for lookup later)
identifier = df_pandas['identifier'].tolist()
nameshop_list = df_pandas['NameShop'].tolist()

In [0]:
def normalize_embeddings(embeddings):
    """
    Normalize each vector to unit length.
    """
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Avoid division by zero
    norms[norms == 0] = 1.0
    return embeddings / norms

normalized_embeddings = normalize_embeddings(embeddings)

In [0]:
# Compute a 2D PCA transformation upfront to speed up plotting.
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(normalized_embeddings)

In [0]:
import umap.umap_ as umap

In [0]:

print("Computing UMAP transformation...")
umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.4, metric='cosine', n_jobs=-1)
umap_model.fit(normalized_embeddings[:20000])  # Train on a smaller subset
embeddings_2d = umap_model.transform(normalized_embeddings)  # Apply to all data
print("UMAP transformation completed.")

In [0]:
# Exact Search with IndexFlatIP
index_flat = faiss.IndexFlatIP(d)
index_flat.add(normalized_embeddings)

# Approximate Search with IndexIVFFlat
nlist = 100  # Number of clusters; tune as needed
quantizer = faiss.IndexFlatIP(d)  # Using inner product metric for quantization
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(normalized_embeddings)  # Training is required
index_ivf.add(normalized_embeddings)
index_ivf.nprobe = 10  # Number of clusters to probe during search

# Approximate Search with HNSW
hnsw_index = faiss.IndexHNSWFlat(d, 32)  # 32 is the number of neighbors in the graph
hnsw_index.hnsw.efConstruction = 40  # Construction parameter (tunable)
hnsw_index.add(normalized_embeddings)


In [0]:
def get_recommendations(query_item_id, index_type="IndexFlatIP", top_n=5, candidate_multiplier=3):
    try:
        query_index = identifier.index(query_item_id)
    except ValueError:
        print(f"Identifier {query_item_id} not found.")
        return None

    query_vector = normalized_embeddings[query_index].reshape(1, -1)
    k = top_n * candidate_multiplier + 1  # Fetch more candidates to avoid returning the same ID
    
    index_map = {
        "IndexFlatIP": index_flat,
        "IndexIVFFlat": index_ivf,
        "IndexHNSWFlat": hnsw_index
    }
    
    if index_type not in index_map:
        print("Invalid index type. Choose from IndexFlatIP, IndexIVFFlat, IndexHNSWFlat.")
        return None
    
    start_time = time.time()
    D, I = index_map[index_type].search(query_vector, k)
    elapsed_time = time.time() - start_time
    
    recs = []
    for sim, idx in zip(D[0], I[0]):
        if idx == query_index:
            continue  # Skip the query item itself
        rec_label = f"{identifier[idx]}:{nameshop_list[idx]}"
        recs.append({'result': rec_label, 'score': float(sim)})
    
    return {'recommendations': recs[:top_n], 'time_taken': elapsed_time}

In [0]:
def get_recommendations(query_item_id, index_type="IndexFlatIP", top_n=5, candidate_multiplier=3):
    try:
        query_index = identifier.index(query_item_id)
    except ValueError:
        print(f"Identifier {query_item_id} not found.")
        return None

    query_vector = normalized_embeddings[query_index].reshape(1, -1)
    k = top_n * candidate_multiplier + 1  # Fetch more candidates to avoid returning the same ID

    index_map = {
        "IndexFlatIP": index_flat,
        "IndexIVFFlat": index_ivf,
        "IndexHNSWFlat": hnsw_index
    }
    
    if index_type not in index_map:
        print("Invalid index type. Choose from IndexFlatIP, IndexIVFFlat, IndexHNSWFlat.")
        return None

    start_time = time.time()
    D, I = index_map[index_type].search(query_vector, k)
    elapsed_time = time.time() - start_time

    seen_ids = set()
    recs = []
    for sim, idx in zip(D[0], I[0]):
        if idx == query_index:
            continue  # Skip the query item itself
        
        # Check if this candidate has already been seen (using identifier as unique key)
        if identifier[idx] in seen_ids:
            continue
        
        seen_ids.add(identifier[idx])
        rec_label = f"{identifier[idx]}:{nameshop_list[idx]}"
        recs.append({'result': rec_label, 'score': float(sim)})
        
        # Stop once we have collected top_n unique recommendations
        if len(recs) == top_n:
            break

    return {'recommendations': recs, 'time_taken': elapsed_time}


In [0]:
query_item = identifier[12]
desired_recommendations = 6
index_type = "IndexIVFFlat"  # Change to "IndexIVFFlat" or "IndexHNSWFlat" as needed
unique_recs = get_recommendations(query_item, index_type=index_type, top_n=desired_recommendations)


In [0]:
unique_recs

In [0]:
def get_unique_recommendations(query_item_id, index_type="IndexFlatIP", desired_count=5, candidate_multiplier=10, max_multiplier=100):
    multiplier = candidate_multiplier
    unique_recs = []
    while multiplier <= max_multiplier and len(unique_recs) < desired_count:
        recs_by_method = get_recommendations(query_item_id, index_type=index_type, top_n=desired_count, candidate_multiplier=multiplier)
        if recs_by_method is None:
            return None
        
        union_dict = {}
        for rec in recs_by_method['recommendations']:
            rec_item_id = rec['result'].split(':')[0]
            score = rec['score']
            if rec_item_id in union_dict:
                if score > union_dict[rec_item_id]['score']:
                    union_dict[rec_item_id] = rec
            else:
                union_dict[rec_item_id] = rec
        
        unique_recs = sorted(union_dict.values(), key=lambda x: x['score'], reverse=True)
        
        if len(unique_recs) >= desired_count:
            return unique_recs[:desired_count]
        multiplier *= 2
    
    if len(unique_recs) < desired_count:
        print(f"Warning: Only {len(unique_recs)} unique recommendations found.")
    
    return unique_recs[:desired_count]


In [0]:
def plot_search_results(query_item_id, unique_recommendations):
    try:
        query_index = identifier.index(query_item_id)
    except ValueError:
        print(f"Item id {query_item_id} not found.")
        return
    
    rec_indices = [identifier.index(rec['result'].split(':')[0]) for rec in unique_recommendations]
    
    plt.figure(figsize=(10, 8))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='grey', alpha=0.2, s=10, label='All products')
    plt.scatter(embeddings_2d[query_index, 0], embeddings_2d[query_index, 1], color='red', s=50, label='Query')
    plt.scatter(embeddings_2d[rec_indices, 0], embeddings_2d[rec_indices, 1], color='blue', s=50, label='Recommendations')
    plt.title("UMAP of Product Embeddings: Query and Unique Recommendations")
    plt.xlabel("UMAP Component 1")
    plt.ylabel("UMAP Component 2")
    plt.legend()
    plt.show()

In [0]:
query_item = identifier[12]
desired_recommendations = 5

In [0]:
query_item = identifier[32]
desired_recommendations = 5
index_type = "IndexIVFFlat"  # Change to "IndexIVFFlat" or "IndexHNSWFlat" as needed
unique_recs = get_unique_recommendations(query_item, index_type=index_type, desired_count=desired_recommendations)


In [0]:
print("Unique Recommendations for item_id:", query_item)
for rec in unique_recs:
    print(rec)

In [0]:
%sql
select distinct * from datascience.at_product_details_individual_embeddings
where identifier in ('02204362','02204362', '06171895', 'A1347042', '01339143', '08608847')
order by array_position(array('02204362','02204362', '06171895', 'A1347042', '01339143', '08608847'), identifier)

In [0]:
plot_search_results(query_item, unique_recs, embeddings_2d)

## using UMAP

In [0]:
# Exact Search with IndexFlatIP
index_flat = faiss.IndexFlatIP(d)
index_flat.add(normalized_embeddings)

# Approximate Search with IndexIVFFlat
nlist = 100  # Number of clusters
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(normalized_embeddings)
index_ivf.add(normalized_embeddings)
index_ivf.nprobe = 10  # Number of clusters to probe

# Approximate Search with HNSW
hnsw_index = faiss.IndexHNSWFlat(d, 32)
hnsw_index.hnsw.efConstruction = 40
hnsw_index.add(normalized_embeddings)

In [0]:
def get_recommendations(query_item_id, index_type="IndexFlatIP", desired_count=5, candidate_multiplier=3, max_multiplier=100):
    """
    Searches for similar items and returns unique recommendations.
    Automatically increases the candidate pool until the desired number
    of unique recommendations is found.
    
    Parameters:
      query_item_id: The item identifier to query.
      index_type: The type of index to use ("IndexFlatIP", "IndexIVFFlat", or "IndexHNSWFlat").
      desired_count: The number of unique recommendations to return.
      candidate_multiplier: Starting multiplier for candidate retrieval.
      max_multiplier: Maximum multiplier allowed to prevent infinite loops.
      
    Returns:
      A dictionary with:
         - 'recommendations': a list of dicts with keys 'result' and 'score'
         - 'time_taken': time taken for the search (from the last iteration)
    """
    try:
        query_index = identifier.index(query_item_id)
    except ValueError:
        print(f"Identifier {query_item_id} not found.")
        return None

    multiplier = candidate_multiplier
    unique_recs = []
    elapsed_time = None

    # Continue increasing the candidate pool until enough unique results are found or we hit max_multiplier
    while multiplier <= max_multiplier:
        query_vector = normalized_embeddings[query_index].reshape(1, -1)
        k = desired_count * multiplier + 1  # +1 to skip the query item itself
        
        index_map = {
            "IndexFlatIP": index_flat,
            "IndexIVFFlat": index_ivf,
            "IndexHNSWFlat": hnsw_index
        }
        
        if index_type not in index_map:
            print("Invalid index type. Choose from IndexFlatIP, IndexIVFFlat, IndexHNSWFlat.")
            return None
        
        start_time = time.time()
        D, I = index_map[index_type].search(query_vector, k)
        elapsed_time = time.time() - start_time
        
        seen_ids = set()
        recs = []
        for sim, idx in zip(D[0], I[0]):
            if idx == query_index:
                continue  # Skip the query item itself
            
            # Check uniqueness based on the identifier (assumed unique)
            if identifier[idx] in seen_ids:
                continue
            
            seen_ids.add(identifier[idx])
            rec_label = f"{identifier[idx]}:{nameshop_list[idx]}"
            recs.append({'result': rec_label, 'score': float(sim)})
            
            # Stop if we've reached the desired count
            if len(recs) == desired_count:
                break
        
        if len(recs) >= desired_count:
            unique_recs = recs[:desired_count]
            break
        else:
            multiplier *= 2  # Increase candidate pool if not enough unique recommendations were found

    if len(unique_recs) < desired_count:
        print(f"Warning: Only {len(unique_recs)} unique recommendations found.")

    return {'recommendations': unique_recs, 'time_taken': elapsed_time}


In [0]:
def plot_search_results(query_item_id, recommendations, embeddings_2d):
    """
    Plots the UMAP visualization of all products, highlighting the query item
    and the recommended items. Also annotates the query item with its name.
    
    Parameters:
      query_item_id: The identifier of the query item.
      recommendations: A list of recommendation dicts from get_recommendations.
      embeddings_2d: 2D embedding coordinates (e.g., from UMAP) for all products.
    """
    try:
        query_index = identifier.index(query_item_id)
    except ValueError:
        print(f"Item id {query_item_id} not found.")
        return
    
    # Get indices of recommended items
    rec_indices = [identifier.index(rec['result'].split(':')[0]) for rec in recommendations]
    
    plt.figure(figsize=(10, 8))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='grey', alpha=0.2, s=10, label='All products')
    plt.scatter(embeddings_2d[query_index, 0], embeddings_2d[query_index, 1], color='red', s=50, label='Query')
    plt.scatter(embeddings_2d[rec_indices, 0], embeddings_2d[rec_indices, 1], color='blue', s=50, label='Recommendations')
    
    # Annotate the query item with its name from nameshop_list (name_ax)
    query_name = nameshop_list[query_index]
    plt.annotate(query_name, 
                 (embeddings_2d[query_index, 0], embeddings_2d[query_index, 1]),
                 textcoords="offset points", xytext=(5, 5), ha='right', fontsize=9, color='red')
    
    plt.title("UMAP of Product Embeddings: Query and Unique Recommendations")
    plt.xlabel("UMAP Component 1")
    plt.ylabel("UMAP Component 2")
    plt.legend()
    plt.show()


In [0]:
# Example usage:
query_item = identifier[30]  # Example query; adjust as needed
desired_recommendations = 5
index_type = "IndexIVFFlat"  # Choose the desired index type

# Get unique recommendations (the function will automatically adjust the candidate pool)
result = get_recommendations(query_item, index_type=index_type, desired_count=desired_recommendations)
if result:
    unique_recs = result['recommendations']
    time_taken = result['time_taken']
    print("Time taken to fetch results:", time_taken, "seconds")
    
    # Print query item id along with its name
    query_name = nameshop_list[identifier.index(query_item)]
    print("Unique Recommendations for item_id: {}:{}".format(query_item, query_name))
    
    # Print each recommendation
    for rec in unique_recs:
        print(rec)
    
    # Plot the results using the provided 2D embeddings
    plot_search_results(query_item, unique_recs, embeddings_2d)

In [0]:
# Example usage:
query_item = identifier[12]  # Example query; adjust as needed
desired_recommendations = 5
index_type = "IndexIVFFlat"  # Choose the desired index type

# Get unique recommendations (the function will automatically adjust the candidate pool)
result = get_recommendations(query_item, index_type=index_type, desired_count=desired_recommendations)
if result:
    unique_recs = result['recommendations']
    time_taken = result['time_taken']
    print("Time taken to fetch results:", time_taken, "seconds")
    
    # Print query item id along with its name
    query_name = nameshop_list[identifier.index(query_item)]
    print("Unique Recommendations for item_id: {}:{}".format(query_item, query_name))
    
    # Print each recommendation
    for rec in unique_recs:
        print(rec)
    
    # Plot the results using the provided 2D embeddings
    plot_search_results(query_item, unique_recs, embeddings_2d)

In [0]:
display([(index, name) for index, name in enumerate(nameshop_list)])